In [33]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor, callback
import lightgbm as lgb

In [34]:
source_file_path = 'data/co2_source.xlsx'
source_sheets = ['Coal', 'Natural gas', 'Petroleum']

sector_file_path = 'data/co2_sector.xlsx'
sector_sheets = ['Residential','Commercial','Industrial','Transportation','Electric power','Total']

other_file_path = 'data/indicator_other.xlsx'
other_sheets = ['Total population','Real GDP','HDD','CDD']

def load_excel_data(path, sheet_list):
    merged_df = None
    for sheet in sheet_list:
        
        df = pd.read_excel(path, sheet_name=sheet, skiprows=2)
        
        
        df_melt = df.melt(id_vars=['State'], var_name='Year', value_name=sheet)
        df_melt['Year'] = pd.to_numeric(df_melt['Year'], errors='coerce')
        
        if merged_df is None:
            merged_df = df_melt
        else:
            merged_df = pd.merge(merged_df, df_melt, on=['State', 'Year'], how='outer')
            
    
    merged_df = merged_df.dropna(subset=['Year'])
    merged_df['Year'] = merged_df['Year'].astype(int)
    merged_df = merged_df[merged_df['State'] != 'US'].dropna()
    return merged_df

df_sector = load_excel_data(sector_file_path, sector_sheets)
df_source = load_excel_data(source_file_path, source_sheets)
df_other = load_excel_data(other_file_path,other_sheets)

df_clean = pd.merge(df_sector, df_source, on=['State', 'Year'], how='inner')
df_clean = pd.merge(df_clean, df_other, on=['State', 'Year'], how='inner')

states = df_clean['State'].unique()

predictions = {}

df_clean['State'] = df_clean['State'].astype('category')
df_clean

,State,Year,Residential,Commercial,Industrial,Transportation,Electric power,Total,Coal,Natural gas,Petroleum,Total population,Real GDP,HDD,CDD
0,AK,1997,1.7,2.5,20.9,13.5,2.7,41.4,1.1,21.9,18.3,613,41071.0,9956,15
1,AK,1998,1.6,2.7,21.4,13.7,2.9,42.4,1.6,22.4,18.5,620,40263.7,10527,10
2,AK,1999,1.9,2.8,20.5,14.8,3.0,43.1,1.6,21.8,19.7,625,39783.1,11591,19
3,AK,2000,1.7,2.7,20.5,15.7,3.2,43.8,1.6,22.6,19.6,628,38428.1,10363,4
4,AK,2001,1.8,2.5,20.6,14.7,3.3,42.8,1.5,21.3,20.0,634,40014.4,10598,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1372,WY,2019,1.0,1.0,12.6,7.9,36.5,59.0,39.2,8.8,10.9,580,38577.1,8906,246
1373,WY,2020,0.9,0.9,11.2,7.2,35.2,55.5,37.2,8.7,9.7,578,36198.2,8069,356
1374,WY,2021,0.9,0.9,11.2,7.4,34.1,54.5,36.1,8.3,10.1,580,37131.8,7865,413
1375,WY,2022,1.0,1.1,11.7,7.2,35.4,56.3,37.4,8.9,10.0,582,37827.2,8441,393


In [35]:
df_60s = df_clean[(df_clean['Year'] >= 1960) & (df_clean['Year'] <= 1969)]
df_70s = df_clean[(df_clean['Year'] >= 1970) & (df_clean['Year'] <= 1979)]
df_80s = df_clean[(df_clean['Year'] >= 1980) & (df_clean['Year'] <= 1989)]
df_90s = df_clean[(df_clean['Year'] >= 1990) & (df_clean['Year'] <= 1999)]
df_00s = df_clean[(df_clean['Year'] >= 2000) & (df_clean['Year'] <= 2009)]
df_10s = df_clean[(df_clean['Year'] >= 2010) & (df_clean['Year'] <= 2019)]

In [36]:
df = pd.read_csv("data/climdiv_county_year.csv")
df = df[["fips", "year", "temp", "tempc"]]
df

,fips,year,temp,tempc
0,1001,1895,62.633333,17.018519
1,1001,1896,65.341667,18.523148
2,1001,1897,65.150000,18.416667
3,1001,1898,63.816667,17.675926
4,1001,1899,63.925000,17.736111
...,...,...,...,...
388370,56045,2015,46.633333,8.129630
388371,56045,2016,47.250000,8.472222
388372,56045,2017,45.650000,7.583333
388373,56045,2018,43.725000,6.513889


In [37]:
df["fips"] = df["fips"].astype(str).str.zfill(5)

In [38]:
df["state_fips"] = df["fips"].str[:2]
df

,fips,year,temp,tempc,state_fips
0,01001,1895,62.633333,17.018519,01
1,01001,1896,65.341667,18.523148,01
2,01001,1897,65.150000,18.416667,01
3,01001,1898,63.816667,17.675926,01
4,01001,1899,63.925000,17.736111,01
...,...,...,...,...,...
388370,56045,2015,46.633333,8.129630,56
388371,56045,2016,47.250000,8.472222,56
388372,56045,2017,45.650000,7.583333,56
388373,56045,2018,43.725000,6.513889,56


In [39]:
state_map = {
    "01": "Alabama",
    "02": "Alaska",
    "04": "Arizona",
    "05": "Arkansas",
    "06": "California",
    "08": "Colorado",
    "09": "Connecticut",
    "10": "Delaware",
    "11": "District of Columbia",
    "12": "Florida",
    "13": "Georgia",
    "15": "Hawaii",
    "16": "Idaho",
    "17": "Illinois",
    "18": "Indiana",
    "19": "Iowa",
    "20": "Kansas",
    "21": "Kentucky",
    "22": "Louisiana",
    "23": "Maine",
    "24": "Maryland",
    "25": "Massachusetts",
    "26": "Michigan",
    "27": "Minnesota",
    "28": "Mississippi",
    "29": "Missouri",
    "30": "Montana",
    "31": "Nebraska",
    "32": "Nevada",
    "33": "New Hampshire",
    "34": "New Jersey",
    "35": "New Mexico",
    "36": "New York",
    "37": "North Carolina",
    "38": "North Dakota",
    "39": "Ohio",
    "40": "Oklahoma",
    "41": "Oregon",
    "42": "Pennsylvania",
    "44": "Rhode Island",
    "45": "South Carolina",
    "46": "South Dakota",
    "47": "Tennessee",
    "48": "Texas",
    "49": "Utah",
    "50": "Vermont",
    "51": "Virginia",
    "53": "Washington",
    "54": "West Virginia",
    "55": "Wisconsin",
    "56": "Wyoming"
}

In [40]:
df["state"] = df["state_fips"].map(state_map)
df = df[["fips", "state_fips", "state", "year", "temp", "tempc"]]
df

,fips,state_fips,state,year,temp,tempc
0,01001,01,Alabama,1895,62.633333,17.018519
1,01001,01,Alabama,1896,65.341667,18.523148
2,01001,01,Alabama,1897,65.150000,18.416667
3,01001,01,Alabama,1898,63.816667,17.675926
4,01001,01,Alabama,1899,63.925000,17.736111
...,...,...,...,...,...,...
388370,56045,56,Wyoming,2015,46.633333,8.129630
388371,56045,56,Wyoming,2016,47.250000,8.472222
388372,56045,56,Wyoming,2017,45.650000,7.583333
388373,56045,56,Wyoming,2018,43.725000,6.513889


<h4> Splitting training and testing dataset and defining independent and dependent variables

In [43]:
train = df_clean.sample(frac=0.7)
test = df_clean[~df_clean.index.isin(train.index)]
ind = ["State", "Year","Total population"]
dep = ["Total"]

# Polynomial Regression

In [42]:
pipeline = Pipeline([
    ("poly", PolynomialFeatures()),
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

param_grid = {
    "poly__degree": [2,3,4,5],
    "poly__interaction_only": [False, True],
    "ridge__alpha": [0.01,0.1,1,10]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="r2")
grid.fit(train[ind], train[dep])

best_model = grid.best_estimator_
dep_predict = best_model.predict(test[ind])

print(grid.best_params_)
print("R² Score:", r2_score(test[dep], dep_predict))
print("MSE:", mean_squared_error(test[dep], dep_predict))
print("MAE:", mean_absolute_error(test[dep], dep_predict))

ValueError: 
All the 160 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
32 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\nicho\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\nicho\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\nicho\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra

# Ridge Regression

In [ ]:
# Features and target
X = df[["year", "state"]]
y = df["temp"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['year']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['state'])
    ]
)

# Ridge pipeline
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RidgeCV(alphas=np.logspace(-3, 3, 20)))
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train
ridge_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = ridge_pipeline.predict(X_test)

print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"Best Alpha: {ridge_pipeline.named_steps['regressor'].alpha_:.4f}")

# Example prediction (Texas 2025)
test_case = pd.DataFrame({
    "year": [2025],
    "state": ["Texas"]
})

pred_temp = ridge_pipeline.predict(test_case)
print(f"\nPredicted 2025 Temperature for Texas: {pred_temp[0]:.2f}")

R² Score: 0.8730
MSE: 9.0246
Best Alpha: 0.1624

Predicted 2025 Temperature for Texas: 65.63


# XGBoost Model

In [44]:
#train["state_fips"] = train["state_fips"].astype("category")
#test["state_fips"] = test["state_fips"].astype("category")
params = {
    "n_estimators": [300, 600, 1000],
    "learning_rate": [0.01, 0.05, 0.1, 0.075],
    "max_depth": [3, 5, 7],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
    "reg_lambda": [1, 1.5, 2],
}

xgb = XGBRegressor(random_state=42, n_jobs=-1, tree_method="hist", enable_categorical=True)
search = RandomizedSearchCV(xgb, params, n_iter=25, scoring="r2", cv=TimeSeriesSplit(n_splits=3), verbose=2, n_jobs=-1)
search.fit(train[ind], train["Total"])
best_model = search.best_estimator_
dep_predict = best_model.predict(test[ind])
print("Best params:", search.best_params_)
print("Best R²:", search.best_score_)
print("MSE:", mean_squared_error(test[dep], dep_predict))
print("MAE:", mean_absolute_error(test[dep], dep_predict))
print("RMSE:", np.sqrt(mean_squared_error(test[dep], dep_predict)))

Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best params: {'subsample': 1.0, 'reg_lambda': 2, 'n_estimators': 1000, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.075, 'gamma': 0, 'colsample_bytree': 1.0}
Best R²: 0.9965549177766561
MSE: 21.346391677856445
MAE: 2.8894338607788086
RMSE: 4.620215544523485


In [ ]:
train["state_fips"] = train["state_fips"].astype("category")
test["state_fips"] = test["state_fips"].astype("category")

train_data = lgb.Dataset(train[ind], label=train["temp"])
test_data = lgb.Dataset(test[ind], label=test["temp"], reference=train_data)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

gbm = lgb.train(params, train_data, num_boost_round=100, valid_sets=[test_data])
dep_predict = gbm.predict(test[ind], num_iteration=gbm.best_iteration)

print("R²:", r2_score(test["temp"], dep_predict))
print("MSE:", mean_squared_error(test["temp"], dep_predict))
print("MAE:", mean_absolute_error(test["temp"], dep_predict))
print("RMSE:", np.sqrt(mean_absolute_error(test["temp"], dep_predict)))

R²: 0.884133590237728
MSE: 8.223316987523507
MAE: 2.184752013482397
RMSE: 1.4780906648383911
